# Tech-Triathlon 2026 — Waypoint Group Datathon

This notebook documents the reproducible final solution for Task 1, Task 2A, and Task 2B. The exact prediction metric is not stated in the official booklet, so the workflow reports multiple transparent chronological metrics and does not describe any surrogate as official.

All predictive models were trained from scratch on the supplied synthetic data. No proprietary modelling API, pretrained predictive model, external dataset, AutoML platform, or low-code/no-code modeller was used.

In [1]:
from pathlib import Path
import json, os, sys, warnings
import joblib
import numpy as np
import pandas as pd

os.environ["LOKY_MAX_CPU_COUNT"] = "16"
warnings.filterwarnings("ignore", message=".*mismatched devices.*")
warnings.filterwarnings("ignore", message="Could not find the number of physical cores.*")

WORK = Path.cwd().resolve()
if not ((WORK / "src").is_dir() and (WORK / "models").is_dir()):
    candidates = [p / "work" for p in [WORK, *WORK.parents] if (p / "work" / "src").exists()]
    if not candidates:
        raise FileNotFoundError("Run this notebook from the extracted submission folder or project work directory")
    WORK = candidates[0]
data_candidates = [p / "Data" for p in [WORK, *WORK.parents] if (p / "Data").is_dir()]
if not data_candidates:
    raise FileNotFoundError("Place the supplied Data folder beside the submission folder")
DATA = data_candidates[0]
PROJECT = DATA.parent
sys.path.insert(0, str(WORK / "src"))

print("Project:", PROJECT)
print("Data:", DATA)
print("Python:", sys.version.split()[0])

Project: E:\UOP\Tech Triathlon
Data: E:\UOP\Tech Triathlon\Data
Python: 3.14.4


## Data inventory and integrity

In [2]:
audit = json.loads((WORK / "reports" / "initial_audit.json").read_text(encoding="utf-8"))
inventory = pd.DataFrame([
    {"file": item["path"], "rows": item["rows"], "columns": len(item["columns"]), "duplicate_rows": item["duplicate_rows"]}
    for item in audit["files"]
])
display(inventory)
print("Join audit:", json.dumps(audit["joins"], indent=2))
print("Template audit:", json.dumps(audit["templates"], indent=2))

,file,rows,columns,duplicate_rows
0,Data\General Data\calendar.csv,910,12,0
1,Data\General Data\district_travel.csv,12,8,0
2,Data\General Data\outlets.csv,120,9,0
3,Data\General Data\road_conditions.csv,10920,3,0
4,Data\General Data\service_allowance.csv,9,3,0
5,Data\General Data\traffic_speed.csv,576,4,0
6,Data\General Data\vehicles.csv,60,9,0
7,Data\Submission Templates\submission_task1.csv,5014,3,0
8,Data\Submission Templates\submission_task2a.csv,60,3,0
9,Data\Submission Templates\submission_task2b.csv,85,6,0


Join audit: {
  "status_counts": {
    "attempted": 90351,
    "deferred": 1543,
    "not_run": 413
  },
  "attempted_rows": 90351,
  "deferred_rows": 1543,
  "dispatched_rows": 91894,
  "train_leg_key_duplicate_rows": 0,
  "train_dispatched_unmatched": 0,
  "train_consistency_mismatches": {
    "outlet": 0,
    "vehicle": 0,
    "brand": 0,
    "district": 0,
    "depot": 0,
    "dispatch_date_vs_leg_date": 0
  },
  "test_order_rows": 5014,
  "test_leg_rows": 5014,
  "test_leg_key_duplicate_rows": 0,
  "test_unmatched": 0,
  "test_outlet_mismatches": 0
}
Template audit: {
  "task1_columns": [
    "delivery_id",
    "pred_service_min",
    "pred_late_prob"
  ],
  "task1_row_count_match": true,
  "task1_order_match": true,
  "task2a_columns": [
    "row_id",
    "pred_total_volume_m3",
    "pred_chilled_volume_m3"
  ],
  "task2a_row_count_match": true,
  "task2a_order_match": true,
  "task2b_columns": [
    "scenario",
    "order_ref",
    "outlet_id",
    "decision",
    "vehicle_id",


## Task 1 label construction

For dispatched orders, `(route_id, seq_in_route)` matches `(route_id, seq)` exactly once. Service begins at `max(actual_arrival, window_open)` so early waiting is excluded. Lateness is a strict comparison: `actual_arrival > window_close`. Actual-time fields are used only here and never enter the model feature matrix.

In [3]:
label_audit = json.loads((WORK / "reports" / "task1_label_audit.json").read_text(encoding="utf-8"))
print("Formula:", label_audit["formula"])
print("Rows:", label_audit["rows"])
print("Early arrivals:", label_audit["early_arrivals"], f"({label_audit['early_arrival_rate']:.2%})")
print("Service median / mean:", label_audit["service_summary"]["50%"], "/", round(label_audit["service_summary"]["mean"], 3))
print("Late rate:", f"{label_audit['late_rate']:.2%}")
print("Arrival exactly at close (non-late):", label_audit["arrival_equal_close_rows"])
assert label_audit["window_cross_midnight_rows"] == 0
assert label_audit["leave_before_arrival_raw_rows"] == 0

Formula: {'service_start': 'max(actual_arrival, window_open)', 'service_min': 'leave_outlet - service_start', 'late': 'actual_arrival > window_close (strict inequality)'}
Rows: 91894
Early arrivals: 4023 (4.38%)
Service median / mean: 15.0 / 19.003
Late rate: 19.58%
Arrival exactly at close (non-late): 279


## Task 1 chronological validation and final GPU models

In [4]:
gpu = json.loads((WORK / "reports" / "task1_gpu_benchmark.json").read_text(encoding="utf-8"))
comparison = pd.DataFrame({
    "model": ["GPU XGBoost depth-6", "Saved CPU booster", "GPU XGBoost loss-guided", "Saved CPU booster"],
    "target": ["service", "service", "lateness", "lateness"],
    "two-fold mean": [
        gpu["scores"]["service"]["xgb_depth6"]["mean_mae"],
        gpu["scores"]["service"]["cpu_existing"]["mean_mae"],
        gpu["scores"]["late"]["xgb_lossguide"]["mean_log_loss"],
        gpu["scores"]["late"]["cpu_existing"]["mean_log_loss"],
    ],
    "metric": ["MAE minutes", "MAE minutes", "log loss", "log loss"],
})
display(comparison)
print("CUDA training confirmed:", gpu["cuda_training_confirmed"])
print("Selected:", gpu["selected_service_model"], "/", gpu["selected_late_model"])

errors = json.loads((WORK / "reports" / "error_analysis.json").read_text(encoding="utf-8"))
display(pd.DataFrame(errors["top_service_features"][:8]).rename(columns={"service_importance_mean": "permutation importance"}))
display(pd.DataFrame(errors["top_late_features"][:8]).rename(columns={"late_importance_mean": "permutation importance"}))

,model,target,two-fold mean,metric
0,GPU XGBoost depth-6,service,4.044108,MAE minutes
1,Saved CPU booster,service,4.060437,MAE minutes
2,GPU XGBoost loss-guided,lateness,0.142450,log loss
3,Saved CPU booster,lateness,0.146201,log loss


CUDA training confirmed: True
Selected: xgb_depth6 / xgb_lossguide


,feature,permutation importance
0,outlet_id,5.323785
1,festival_ramp,1.642673
2,order_units,0.959113
3,is_payday,0.274015
4,dow,0.263862
5,order_weight_kg,0.161132
6,order_volume_m3,0.132805
7,planned_margin_to_close_min,0.065024


,feature,permutation importance
0,planned_margin_to_close_min,0.195258
1,outlet_id,0.098730
2,disruption_index,0.042737
3,festival_ramp,0.040801
4,seq_in_route,0.014068
5,dow,0.012112
6,planned_arrival_min,0.008699
7,route_total_planned_travel_min,0.008140


## Task 2A weekly demand forecasting

Training demand combines every order from the training file and Task 1 test inputs, including deferred and not-run orders, and assigns volume to the requested ISO week. Lag and rolling features are shifted. Future calendar fields are known from the supplied calendar. Every backtest recursively predicts the next 10 weeks.

In [5]:
task2a_report = json.loads((WORK / "reports" / "task2a_model_report.json").read_text(encoding="utf-8"))
rows = []
for target, detail in task2a_report["targets"].items():
    for model, score in detail["candidates"].items():
        rows.append({"target": target, "model": model, "mean WAPE": score["mean_wape"], "latest WAPE": score["latest_wape"], "selected": model == detail["selected"]})
display(pd.DataFrame(rows).sort_values(["target", "mean WAPE"]))
print("Forecast horizon:", task2a_report["forecast_start"], "to", task2a_report["forecast_end"])
print("Selected models:", task2a_report["selected"])
assert task2a_report["missing_series_weeks"] == 0

,target,model,mean WAPE,latest WAPE,selected
14,chilled_volume_m3,hgb,0.039695,0.031660,True
13,chilled_volume_m3,ridge,0.039925,0.039686,False
15,chilled_volume_m3,hgb_log,0.041454,0.033332,False
8,chilled_volume_m3,historical_mean,0.050527,0.040198,False
10,chilled_volume_m3,rolling13,0.052458,0.040919,False
9,chilled_volume_m3,rolling4,0.055273,0.053771,False
12,chilled_volume_m3,seasonal_recent_blend,0.055952,0.040942,False
11,chilled_volume_m3,seasonal52,0.068607,0.050180,False
6,total_volume_m3,hgb,0.053232,0.046833,True
2,total_volume_m3,rolling13,0.060401,0.056001,False


Forecast horizon: 2026-03-30 to 2026-06-01
Selected models: {'total_volume_m3': 'hgb', 'chilled_volume_m3': 'hgb'}


## Task 2B allocation and official validation

In [6]:
allocation = json.loads((WORK / "reports" / "task2b_optimization_report.json").read_text(encoding="utf-8"))
final_audit = json.loads((WORK / "reports" / "final_audit.json").read_text(encoding="utf-8"))
display(pd.DataFrame(allocation["sensitivity"]).T[["served_orders", "deferred_orders", "deferred_yesterday_served", "days_since_last_served_sum", "chilled_orders_served"]])
print("Selected policy:", allocation["selected_policy"])
print("Maximum feasible order count:", allocation["selected_summary"]["maximum_feasible_order_count"])
print("Official validator:", final_audit["task2b_validator_output"])
assert final_audit["task2b_official_validator"]

,served_orders,deferred_orders,deferred_yesterday_served,days_since_last_served_sum,chilled_orders_served
fairness_first,77,8,10,130,19
cold_chain,77,8,10,130,19
coverage,77,8,10,130,19
max_orders,79,6,9,128,21


Selected policy: fairness_first
Maximum feasible order count: 79
Official validator: FEASIBILITY: PASSED - every rule satisfied.


## Final saved-model inference demonstration

In [7]:
# FINAL CELL: load the saved models and run inference without retraining.
from inference import predict_task1, predict_task2a

task1_bundle = joblib.load(WORK / "models" / "task1_models.pkl")
task2a_bundle = joblib.load(WORK / "models" / "task2a_models.pkl")

task1_inputs = pd.read_csv(DATA / "Test Data" / "task1_test_inputs.csv").head(5)
task1_predictions = predict_task1().head(5)
print("TASK 1 REPRESENTATIVE INPUTS")
display(task1_inputs[["delivery_id", "outlet_id", "brand", "order_volume_m3", "planned_arrival_time", "window_close_time"]])
print("TASK 1 PREDICTIONS")
display(task1_predictions)

task2a_inputs = pd.read_csv(DATA / "Test Data" / "task2a_test_inputs.csv").head(6)
task2a_predictions = predict_task2a().head(6)
print("TASK 2A REPRESENTATIVE INPUTS")
display(task2a_inputs)
print("TASK 2A PREDICTIONS")
display(task2a_predictions)

print("Loaded Task 1 models:", task1_bundle["selected_service_model"], "/", task1_bundle["selected_late_model"])
print("Loaded Task 2A models:", task2a_bundle["selected"])
print("Inference completed from saved model files.")

TASK 1 REPRESENTATIVE INPUTS


,delivery_id,outlet_id,brand,order_volume_m3,planned_arrival_time,window_close_time
0,ORD0092308,OUT001,Fresh,0.501,05:21,07:30
1,ORD0092309,OUT002,Fresh,0.539,06:10,08:00
2,ORD0092310,OUT003,Fresh,0.472,05:46,07:30
3,ORD0092311,OUT004,Fresh,1.193,06:44,08:00
4,ORD0092312,OUT005,Fresh,2.687,06:22,07:45


TASK 1 PREDICTIONS


,delivery_id,pred_service_min,pred_late_prob
0,ORD0092308,8.361660,0.000155
1,ORD0092309,8.067531,0.000530
2,ORD0092310,12.087658,0.000319
3,ORD0092311,11.857336,0.004319
4,ORD0092312,14.509268,0.008601


  File "C:\Users\prabh\AppData\Roaming\Python\Python314\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
  File "C:\Users\prabh\AppData\Roaming\Python\Python314\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Python314\Lib\subprocess.py", line 555, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python314\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
       

TASK 2A REPRESENTATIVE INPUTS


,row_id,depot,brand,iso_year,iso_week
0,W0000,Kandy,Fresh,2026,14
1,W0001,Kandy,Style,2026,14
2,W0002,Kandy,Tech,2026,14
3,W0003,Peliyagoda,Fresh,2026,14
4,W0004,Peliyagoda,Style,2026,14
5,W0005,Peliyagoda,Tech,2026,14


TASK 2A PREDICTIONS


,row_id,pred_total_volume_m3,pred_chilled_volume_m3
0,W0000,532.123058,200.680733
1,W0001,77.883468,0.000000
2,W0002,23.379806,0.000000
3,W0003,1003.387134,380.212082
4,W0004,153.292793,0.000000
5,W0005,34.997703,0.000000


Loaded Task 1 models: xgb_depth6 / xgb_lossguide
Loaded Task 2A models: {'total_volume_m3': 'hgb', 'chilled_volume_m3': 'hgb'}
Inference completed from saved model files.
